# STAGE 9B-cal · Calibrating λ before the expensive run

### ~15 min · ~0.5 CU · `best.pt` never written

The smoke test showed the adversary's loss **falling** (`Lp` 0.683 → 0.619) and projection AUC plateauing at 0.82. Either λ is too weak, or 36 steps was simply too few to tell. **This resolves it for ~0.5 CU instead of 4.**

## Why your setup is harder than Pereira's

| | Pereira | you |
|---|---|---|
| training steps | ~22,000 | 4,542 (**5× fewer**) |
| starting point | ImageNet | **converged checkpoint** — projection already deeply encoded |

Same adversarial step size, far less opportunity to apply it.

## What we look for

| signal | meaning |
|---|---|
| **`Lp` rising** | the adversary is being starved — **the reversal is biting** |
| **projAUC falling** | projection information is leaving the features |
| AUROC holding | the disease head survives |

Pereira reached projection AUC **0.61** (from 0.99). We want the smallest λ that moves meaningfully toward that without collapsing AUROC.

---
# 0 · Setup

In [ ]:
import os, sys, json, time, hashlib, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch

LAMBDAS   = [0.2, 0.5, 1.0, 2.0]   # 0.2 = the smoke-test value
N_TRAIN   = 6000                   # subset: enough steps to see the trend
EPOCHS    = 3
LR_BACKBONE, LR_ADV = 5e-5, 1e-4
BATCH, NUM_WORKERS  = 48, 2

from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
else: print('  Drive already mounted')
PROJECT  = Path('/content/drive/MyDrive/Component_01')
IMG_ROOT = Path('/content/cardio_image_384')
TAR      = PROJECT / 'data' / 'images' / 'cardio_384.tar'
MANIFEST = PROJECT / 'training_manifest'
S5_CKPT  = PROJECT / 'checkpoints' / 'stage5' / 'best.pt'
OUT      = PROJECT / 'reports' / 'stage9b'; OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT))
SHA_BEFORE = hashlib.sha256(S5_CKPT.read_bytes()).hexdigest()
print('  best.pt SHA-256:', SHA_BEFORE[:40])
DEV = 'cuda'; assert torch.cuda.is_available(), 'select an L4 GPU'

if not IMG_ROOT.exists():
    import subprocess, shutil
    t0 = time.time(); lt = Path('/content/cardio_384.tar')
    if not lt.exists(): shutil.copy(TAR, lt)
    subprocess.run(['tar', '-xf', str(lt), '-C', '/content'], check=True)
    print('  staged in %.1f min' % ((time.time() - t0) / 60))
print('  images:', IMG_ROOT.exists())

---
# 1 · Data & model factory

In [ ]:
import stage6_acr as acr, stage9b_gradrev as s9b
from cxr_transforms import build_transform
from torch.utils.data import DataLoader
PATH = s9b.PATHOLOGIES
cfg  = json.loads((MANIFEST / 'manifest_config.json').read_text())
pw   = torch.tensor([min(cfg['pos_weight'][k], 8.0) for k in PATH], dtype=torch.float32)

tr = pd.read_csv(MANIFEST / 'manifest_train.csv', low_memory=False)
# Stratified subset: preserve the AP/PA ratio, or the adversary faces a
# different problem from the full run and the calibration will not transfer.
frac = N_TRAIN / len(tr)
tr = (tr.groupby('view', group_keys=False)
        .apply(lambda g: g.sample(max(int(round(len(g) * frac)), 1), random_state=0)))
va = pd.read_csv(MANIFEST / 'manifest_val.csv', low_memory=False).sample(2000, random_state=0)
print('  train %d (AP %d / PA %d)   val %d'
      % (len(tr), (tr.view == 'AP').sum(), (tr.view == 'PA').sum(), len(va)))

TF_TR, TF_EV = build_transform('train'), build_transform('test')
dl_tr = DataLoader(s9b.CXRDataset(tr, IMG_ROOT, TF_TR, PATH), batch_size=BATCH,
                   shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
dl_va = DataLoader(s9b.CXRDataset(va, IMG_ROOT, TF_EV, PATH), batch_size=BATCH * 2,
                   shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
STEPS = len(dl_tr) * EPOCHS
print('  %d steps per lambda  x %d lambdas' % (STEPS, len(LAMBDAS)))

CK = torch.load(S5_CKPT, map_location='cpu', weights_only=False)   # READ ONLY, once
crit_d = s9b.WeightedBCE(pw).to(DEV)

def fresh_model():
    m = s9b.CXRGradRev(len(PATH))
    m.load_stage5(CK, use_ema=True)
    return m.to(DEV).to(memory_format=torch.channels_last)

@torch.no_grad()
def evaluate(model):
    model.eval(); D, Y, A, PJ = [], [], [], []
    for x, y, w, a in dl_va:
        x = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            d, p = model(x, None, lambd=0.0)
        D.append(torch.sigmoid(d.float()).cpu().numpy())
        PJ.append(torch.sigmoid(p.float()).cpu().numpy())
        Y.append(y.numpy()); A.append(a.numpy())
    P = pd.DataFrame(np.concatenate(D), columns=PATH)
    L = pd.DataFrame(np.concatenate(Y).astype(int), columns=PATH)
    ap = np.concatenate(A) > 0.5; proj = np.concatenate(PJ)
    au = [acr.auroc(L[k], P[k]) for k in PATH]
    gp = [acr.auroc(L[k][~ap], P[k][~ap]) - acr.auroc(L[k][ap], P[k][ap]) for k in PATH]
    model.train()
    return dict(auroc=float(np.nanmean(au)), gap=float(np.nanmean(gp)),
                proj_auc=float(acr.auroc(ap.astype(int), proj)))

---
# 2 · The sweep ★

Each λ starts from a **fresh copy of `best.pt`**, so the four runs are independent.

In [ ]:
from tqdm.auto import tqdm
RES = {}
T0 = time.time()
for lam_max in LAMBDAS:
    model = fresh_model()
    opt = torch.optim.AdamW(model.param_groups(LR_BACKBONE, LR_ADV), weight_decay=0.05)
    gstep = 0
    traj = [dict(epoch=0, Ld=float('nan'), Lp=float('nan'), **evaluate(model))]
    for ep in range(EPOCHS):
        model.train(); rd = rp = 0.0; n = 0
        bar = tqdm(dl_tr, desc='lam=%.1f ep%d' % (lam_max, ep + 1), leave=False)
        for x, y, w, a in bar:
            x = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
            y, w, a = y.to(DEV), w.to(DEV), a.to(DEV)
            lam = s9b.lambda_at(gstep, STEPS, lam_max)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                d, p = model(x, y, lambd=lam)
            ld = crit_d(d.float(), y, w)
            lp = s9b.projection_loss(p.float(), a)
            loss = ld + lp
            assert torch.isfinite(loss), 'non-finite loss at lambda %.1f' % lam_max
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); gstep += 1
            rd += ld.item(); rp += lp.item(); n += 1
            bar.set_postfix(Ld='%.3f' % (rd / n), Lp='%.3f' % (rp / n))
        m = evaluate(model)
        m.update(epoch=ep + 1, Ld=rd / n, Lp=rp / n)
        traj.append(m)
        print('  lam %.1f  ep%d  Ld %.4f  Lp %.4f | AUROC %.4f  gap %.4f  projAUC %.4f'
              % (lam_max, ep + 1, m['Ld'], m['Lp'], m['auroc'], m['gap'], m['proj_auc']))
    RES[lam_max] = traj
    del model, opt; torch.cuda.empty_cache()
    print()
print('  sweep done in %.1f min' % ((time.time() - T0) / 60))

---
# 3 · Verdict

In [ ]:
print('=' * 88)
print('  LAMBDA SWEEP   (val n=%d, %d steps per lambda)' % (len(va), STEPS))
print('=' * 88)
print('  %8s %10s %10s %12s %11s %10s  %s'
      % ('lambda', 'Lp start', 'Lp end', 'Lp change', 'projAUC', 'AUROC', 'verdict'))
print('  ' + '-' * 85)
best_lam, best_score = None, -1e9
for lam in LAMBDAS:
    t = RES[lam]
    lp0, lp1 = t[1]['Lp'], t[-1]['Lp']
    pa, au = t[-1]['proj_auc'], t[-1]['auroc']
    bite = 'BITING' if lp1 > lp0 else 'no'
    print('  %8.1f %10.4f %10.4f %+12.4f %11.4f %10.4f  %s'
          % (lam, lp0, lp1, lp1 - lp0, pa, au, bite))
    # prefer low projection AUC; penalise any collapse of disease AUROC
    score = (0.85 - pa) - max(0.0, 0.84 - au) * 5.0
    if score > best_score: best_score, best_lam = score, lam
print('  ' + '-' * 85)
print()
print('  Reference: Pereira reached projection AUC 0.61 (from 0.99).')
print()
biting = [l for l in LAMBDAS if RES[l][-1]['Lp'] > RES[l][1]['Lp']]
if biting:
    print('  Reversal BITES at lambda:', biting)
    print('  RECOMMENDED LAMBDA_MAX for the full run: %.1f' % best_lam)
    print('  Set LAMBDA_MAX in Stage9B_Gradient_Reversal.ipynb and run it.')
else:
    print('  The reversal does NOT bite at any tested lambda.')
    print('  Fine-tuning from a converged checkpoint leaves too few steps to')
    print('  unlearn projection. Options:')
    print('    (a) larger lambda (extend LAMBDAS and re-run this sweep)')
    print('    (b) train from ImageNet like Pereira (~20 CU)')
    print('    (c) report the failure to reproduce as the finding')

(OUT / 'stage9b_lambda_sweep.json').write_text(
    json.dumps({str(k): v for k, v in RES.items()}, indent=2, default=float),
    encoding='utf-8')
print('\n  saved', OUT / 'stage9b_lambda_sweep.json')
assert hashlib.sha256(S5_CKPT.read_bytes()).hexdigest() == SHA_BEFORE, 'best.pt MODIFIED!'
print('  *** best.pt VERIFIED BYTE-IDENTICAL ***')